### Spotify Analysis

In [1]:
!pip install spotipy

In [2]:
import requests
key = []

with open('C:/Users/wooju/Desktop/key.txt', 'r', encoding='utf-8') as file:
    for row in file:
        key.append(row.rstrip('\n'))

CLIENT_ID = key[0]
CLIENT_PW = key[1]

In [3]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
client_credentials_manager = SpotifyClientCredentials(client_id=CLIENT_ID, client_secret=CLIENT_PW)
sp = spotipy.Spotify(client_credentials_manager=client_credentials_manager)

In [4]:
artist_name = []
track_name = []
track_popularity = []
artist_id = []
track_id = []

for i in range(0, 1000, 50):
    track_results = sp.search(q='year:2022', type = 'track', limit = 50, offset = 1)
    for i, t in enumerate(track_results['tracks']['items']):
        artist_name.append(t['artists'][0]['name'])
        artist_id.append(t['artists'][0]['id'])
        track_name.append(t['name'])
        track_id.append(t['id'])
        track_popularity.append(t['popularity'])

In [7]:
import pandas as pd
track_df = pd.DataFrame({'artist_name' : artist_name, 'track_name' : track_name, 'track_id' : track_id, 'track_popularity' : track_popularity, 'artist_id' : artist_id})
print(track_df.shape)
track_df.head(4)

(1000, 5)


,artist_name,track_name,track_id,track_popularity,artist_id
0,Ish Kevin,My Year (2022),16TeLUubmneRAhwmRAMbjM,22,27rjDVGfZG4eorQwrZt2ZD
1,Alonestar,COOKING UP,5D87MkcNRiZ4AsVMwxRrC1,34,1kwi1Q7X4jPTkHTgayxKhS
2,Mabel,Loneliest Time Of Year,66QVjmCkXqtYx3YRgEoj1n,1,1MIVXf74SZHmTIp4V4paH4
3,Harry Styles,As It Was,4LRPiXqCikLlN15c3yImP7,100,6KImCVD70vtIoJWnq6nGn3


In [49]:
track_features = []

for t_id in track_df['track_id']:
    af = sp.audio_features(t_id)
    track_features.append(af[0])
tf_df = pd.DataFrame(columns = ['danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'type', 'id', 'uri', 'track_href', 'analysis_url', 'duration_ms', 'time_signature'])

for item in track_features:
    sr = pd.Series(item)
    df = pd.DataFrame(sr).T
    tf_df = pd.concat([tf_df, df], axis = 0)
tf_df.head()

,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,type,id,uri,track_href,analysis_url,duration_ms,time_signature
0,0.521,0.683,5,-6.24,0,0.501,0.159,0,0.187,0.737,139.439,audio_features,16TeLUubmneRAhwmRAMbjM,spotify:track:16TeLUubmneRAhwmRAMbjM,https://api.spotify.com/v1/tracks/16TeLUubmneR...,https://api.spotify.com/v1/audio-analysis/16Te...,161808,3
0,0.544,0.792,2,-8.73,1,0.343,0.261,0,0.267,0.628,87.891,audio_features,5D87MkcNRiZ4AsVMwxRrC1,spotify:track:5D87MkcNRiZ4AsVMwxRrC1,https://api.spotify.com/v1/tracks/5D87MkcNRiZ4...,https://api.spotify.com/v1/audio-analysis/5D87...,211766,4
0,0.517,0.609,7,-5.702,1,0.0291,0.0631,0,0.106,0.239,92.093,audio_features,66QVjmCkXqtYx3YRgEoj1n,spotify:track:66QVjmCkXqtYx3YRgEoj1n,https://api.spotify.com/v1/tracks/66QVjmCkXqtY...,https://api.spotify.com/v1/audio-analysis/66QV...,202693,4
0,0.52,0.731,6,-5.338,0,0.0557,0.342,0.00101,0.311,0.662,173.93,audio_features,4LRPiXqCikLlN15c3yImP7,spotify:track:4LRPiXqCikLlN15c3yImP7,https://api.spotify.com/v1/tracks/4LRPiXqCikLl...,https://api.spotify.com/v1/audio-analysis/4LRP...,167303,4
0,0.437,0.605,8,-5.5,0,0.0329,0.00924,0.118,0.264,0.581,125.114,audio_features,3LdLGNKCW0dNr14JQlZPQt,spotify:track:3LdLGNKCW0dNr14JQlZPQt,https://api.spotify.com/v1/tracks/3LdLGNKCW0dN...,https://api.spotify.com/v1/audio-analysis/3LdL...,191160,4


In [50]:
cols_to_drop2 = ['key','mode','type','uri','track_href','analysis_url']
tf_df.drop(columns=cols_to_drop2, inplace=True)
print(tf_df.info())

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1000 entries, 0 to 0
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   danceability      1000 non-null   object
 1   energy            1000 non-null   object
 2   loudness          1000 non-null   object
 3   speechiness       1000 non-null   object
 4   acousticness      1000 non-null   object
 5   instrumentalness  1000 non-null   object
 6   liveness          1000 non-null   object
 7   valence           1000 non-null   object
 8   tempo             1000 non-null   object
 9   id                1000 non-null   object
 10  duration_ms       1000 non-null   object
 11  time_signature    1000 non-null   object
dtypes: object(12)
memory usage: 101.6+ KB
None


In [51]:
artist_popularity = []
artist_genres = []
artist_followers = []
for a_id in track_df['artist_id']:
    artist = sp.artist(a_id)
    artist_popularity.append(artist['popularity'])
    artist_genres.append(artist['genres'])
    artist_followers.append(artist['followers']['total'])

In [52]:
track_df = track_df.assign(artist_popularity = artist_popularity, artist_genres = artist_genres, artist_followers = artist_followers)
track_df.head()

,artist_name,track_name,track_id,track_popularity,artist_id,artist_popularity,artist_genres,artist_followers
0,Ish Kevin,My Year (2022),16TeLUubmneRAhwmRAMbjM,22,27rjDVGfZG4eorQwrZt2ZD,23,[rwandan hip hop],6803
1,Alonestar,COOKING UP,5D87MkcNRiZ4AsVMwxRrC1,34,1kwi1Q7X4jPTkHTgayxKhS,50,[],3874
2,Mabel,Loneliest Time Of Year,66QVjmCkXqtYx3YRgEoj1n,1,1MIVXf74SZHmTIp4V4paH4,70,"[dance pop, edm, pop, pop dance, post-teen pop...",1624624
3,Harry Styles,As It Was,4LRPiXqCikLlN15c3yImP7,100,6KImCVD70vtIoJWnq6nGn3,93,[pop],21671122
4,Zager & Evans,In the Year 2525,3LdLGNKCW0dNr14JQlZPQt,38,55IyYO6fmYpYw6Nd4YF7bw,35,"[bubblegum pop, folk rock]",8385


In [53]:
print(track_df.info())
print(tf_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   artist_name        1000 non-null   string
 1   track_name         1000 non-null   string
 2   track_id           1000 non-null   string
 3   track_popularity   1000 non-null   int64 
 4   artist_id          1000 non-null   string
 5   artist_popularity  1000 non-null   int64 
 6   artist_genres      1000 non-null   object
 7   artist_followers   1000 non-null   int64 
dtypes: int64(3), object(1), string(4)
memory usage: 62.6+ KB
None
<class 'pandas.core.frame.DataFrame'>
Int64Index: 1000 entries, 0 to 0
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   danceability      1000 non-null   object
 1   energy            1000 non-null   object
 2   loudness          1000 non-null   object
 3   speechiness       1

In [54]:
track_df['artist_name'] = track_df['artist_name'].astype('string')
track_df['track_name'] = track_df['track_name'].astype('string')
track_df['track_id'] = track_df['track_id'].astype('string')
track_df['artist_id'] = track_df['artist_id'].astype('string')
tf_df['duration_ms'] = pd.to_numeric(tf_df['duration_ms'])
tf_df['instrumentalness'] = pd.to_numeric(tf_df['instrumentalness'])
tf_df['time_signature'] = tf_df['time_signature'].astype('category')
print(track_df.info())
print(tf_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   artist_name        1000 non-null   string
 1   track_name         1000 non-null   string
 2   track_id           1000 non-null   string
 3   track_popularity   1000 non-null   int64 
 4   artist_id          1000 non-null   string
 5   artist_popularity  1000 non-null   int64 
 6   artist_genres      1000 non-null   object
 7   artist_followers   1000 non-null   int64 
dtypes: int64(3), object(1), string(4)
memory usage: 62.6+ KB
None
<class 'pandas.core.frame.DataFrame'>
Int64Index: 1000 entries, 0 to 0
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   danceability      1000 non-null   object  
 1   energy            1000 non-null   object  
 2   loudness          1000 non-null   object  
 3   speechine

In [55]:
track_df.head()

,artist_name,track_name,track_id,track_popularity,artist_id,artist_popularity,artist_genres,artist_followers
0,Ish Kevin,My Year (2022),16TeLUubmneRAhwmRAMbjM,22,27rjDVGfZG4eorQwrZt2ZD,23,[rwandan hip hop],6803
1,Alonestar,COOKING UP,5D87MkcNRiZ4AsVMwxRrC1,34,1kwi1Q7X4jPTkHTgayxKhS,50,[],3874
2,Mabel,Loneliest Time Of Year,66QVjmCkXqtYx3YRgEoj1n,1,1MIVXf74SZHmTIp4V4paH4,70,"[dance pop, edm, pop, pop dance, post-teen pop...",1624624
3,Harry Styles,As It Was,4LRPiXqCikLlN15c3yImP7,100,6KImCVD70vtIoJWnq6nGn3,93,[pop],21671122
4,Zager & Evans,In the Year 2525,3LdLGNKCW0dNr14JQlZPQt,38,55IyYO6fmYpYw6Nd4YF7bw,35,"[bubblegum pop, folk rock]",8385


In [56]:
tf_df.head()

,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,id,duration_ms,time_signature
0,0.521,0.683,-6.24,0.501,0.159,0.00000,0.187,0.737,139.439,16TeLUubmneRAhwmRAMbjM,161808,3
0,0.544,0.792,-8.73,0.343,0.261,0.00000,0.267,0.628,87.891,5D87MkcNRiZ4AsVMwxRrC1,211766,4
0,0.517,0.609,-5.702,0.0291,0.0631,0.00000,0.106,0.239,92.093,66QVjmCkXqtYx3YRgEoj1n,202693,4
0,0.52,0.731,-5.338,0.0557,0.342,0.00101,0.311,0.662,173.93,4LRPiXqCikLlN15c3yImP7,167303,4
0,0.437,0.605,-5.5,0.0329,0.00924,0.11800,0.264,0.581,125.114,3LdLGNKCW0dNr14JQlZPQt,191160,4
